In [ ]:
import requests, json, time, random, os
import pandas as pd
from config import DEEPSEEK_KEY

n_sim=1000

df_raw_all=pd.read_csv('report_conclusions_raw.csv')
df_sample=df_raw_all[:1000]
# df_sample.to_excel('human_conclusion.xlsx')

In [ ]:
# from openai import OpenAI
client = OpenAI(api_key=DEEPSEEK_KEY, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一名专业的券商分析师。"},
        {"role": "user", "content": "你好。"},
    ],
    stream=False
)

print(response.choices[0].message.content)

您好！很高兴为您服务。作为专业的券商分析师，我可以为您提供市场动态、行业研究、投资策略等方面的专业分析。如果您有任何问题或需要帮助，请随时告诉我！


In [ ]:
for i in range(n_sim):
    call_llm(df_sample['Conclusion'][i])

In [ ]:
df_raw_all = pd.read_csv('report_conclusions_raw.csv')
df_sample = df_raw_all[:1000]


In [11]:
df_sample.head()

,Unnamed: 0,WritingDate,Title,Conclusion,OrgNameDisc
0,0,2025-05-12,汉朔科技(301275)：电子价签行业领跑者 助力泛零售数字化升级,核心结论：我们预计：公司有望在2025 年-2027 年分别实现归母净利润8.18亿元、...,西部证券
1,1,2025-05-11,科大智能(300222)：战略聚焦数字能源 拓展智能机器人应用,公司深耕电力能源领域，聚焦数字能源战略成效初显。公司成立于2002 年，是一家深耕电力能...,广发证券
2,2,2025-05-09,海晨股份(300873)：业内稀缺跨领域物流方案提供商 AMHS业务构筑第二成长曲线,业内稀缺一体化供应链解决方案领军企业，下游优质客户增厚公司业绩弹性。公司长期聚焦服务于消...,民生证券
3,3,2025-05-08,川环科技(300547)：汽车管路龙头 进军AIDC液冷可期,公司基本情况：公司成立于2002 年，自设立以来一直专注于车用胶管系列产品，主营业务包括...,天风证券
4,4,2025-05-09,金力永磁(300748)：秉技术优势 乘行业东风 迎跨越发展,金力永磁是全球领先的高性能稀土永磁材料龙头企业。纵向来看，公司产能从2021年的1.5 ...,东方证券


In [ ]:
DEEPSEEK_URL = "https://api.deepseek.com/chat/completions"

# 您的原始提示词
prompt_2="""你是一名专业的券商分析师。
请参考输入的文本，仿照按照真实券商研究报告的风格，中文撰写一个类似的片段。
但不能照搬，也不能整合改编给你的输入。而是关于一个不同的公司的。
给出的报告长度接近。"""

prompt_3="""你是一名专业的券商分析师。
请参考输入的文本，仿照按照真实券商研究报告的风格, 中文撰写一个类似的片段。
整合改编给你的输入, 进行润色, 但不要一致。
给出的报告长度接近。"""


def call_deepseek(input_text, prompt):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {DEEPSEEK_KEY}"
    }
    
    data = {
        "model": "deepseek-chat",  # 使用非思考模式，响应更快[citation:1]
        "messages": [
            {
                "role": "system", 
                "content": "You are a professional financial analyst."
            },
            {
                "role": "user",
                "content": f"{prompt}\n\n输入文本: {input_text}"
            }
        ],
        "stream": False,
        "temperature": 0.1,  # 稍低的温度值以获得更稳定的输出
        "max_tokens": 4096   # 根据需求调整输出长度
    }
    
    try:
        response = requests.post(
            DEEPSEEK_URL, 
            headers=headers, 
            data=json.dumps(data),
            timeout=60  # 60秒超时
        )
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"DeepSeek API调用失败, 状态码: {response.status_code}")
            print(f"错误信息: {response.text}")
            return None
    except Exception as e:
        print(f"DeepSeek API调用异常: {e}")
        return None

In [32]:
df_sample=df_sample[100:].reset_index()
df_sample.head()

,index,Unnamed: 0,WritingDate,Title,Conclusion,OrgNameDisc
0,300,300,2025-02-07,匠心家居(301061)：表现超预期 产品品牌升级、汇兑共振,公司发布2024 年业绩预告，业绩超预期。预计2024 年实现归母净利5.70-6.50...,申万宏源研究
1,301,301,2025-02-07,特锐德(300001)：充电桩产业链龙头 AI基建驱动箱变高成长,核心观点：\r\n 变配电“智能制造+系统集成”根基稳固，充电网业务贡献第二成长曲线。...,广发证券
2,302,302,2025-02-07,运达股份(300772)：2025年出货量有望较快增长 风机盈利有提升空间,公司近况\r\n 2024 年国内新增风电装机约88GW（CWEA统计），我们预计20...,中金公司
3,303,303,2025-02-07,南山智尚(300918)：积极推进UHMWPE业务在机器人领域的应用 未来发展潜力大,核心观点：\r\n 公司正在积极推进UHMWPE 业务在机器人领域相关下游的应用。根据...,广发证券
4,304,304,2025-02-06,卓胜微(300782)：拟募集不超过35亿元扩建射频制造产线 25年发射端模组L-PAMID...,事件：拟募集不超过35 亿元，扩建关键射频芯片制造产线，扩充产能瓶颈。\r\n 1 月...,长城证券


In [ ]:
# 批量处理代码
n_sim = 1000

# 读取数据
final_result=[]
import time
start_time=time.time()
# 添加延迟控制，避免速率限制
for i in range(min(n_sim, len(df_sample))):
    result = call_deepseek(df_sample['Conclusion'][i],prompt_2)
    final_result.append(result)
    
    now=time.time()
    # 处理结果，这里可以根据需要保存结果
    if result:
        print(f"第{i+1}条处理完成")
    else:
        print(f"第{i+1}条处理失败")
    print(now-start_time)
    # 添加延迟避免触发速率限制
    if (i + 1) % 10 == 0:
        time.sleep(1)
    
    if i%20==0:
        pd.DataFrame(final_result).to_excel(fr'gen_1027{time.time()}.xlsx')

第1条处理完成
18.09815788269043
第2条处理完成
40.02473425865173
第3条处理完成
61.779582500457764
第4条处理完成
80.22168326377869
第5条处理完成
102.4254081249237
第6条处理完成
123.79368042945862
第7条处理完成
144.40318655967712
第8条处理完成
171.37120151519775
第9条处理完成
195.75792860984802
第10条处理完成
211.48763990402222
第11条处理完成
236.43943858146667
第12条处理完成
269.86166048049927
第13条处理完成
300.5595209598541
第14条处理完成
316.92189955711365
第15条处理完成
336.1926853656769
第16条处理完成
356.26495695114136
第17条处理完成
380.68309116363525
第18条处理完成
398.37893867492676
第19条处理完成
417.20757246017456
第20条处理完成
443.3310823440552
第21条处理完成
469.876832485199
第22条处理完成
485.8290903568268
第23条处理完成
505.44957661628723
第24条处理完成
532.4462614059448
第25条处理完成
549.1586399078369
第26条处理完成
569.2184052467346
第27条处理完成
591.711995601654
第28条处理完成
622.8784649372101
第29条处理完成
648.4093112945557
第30条处理完成
667.6751036643982
第31条处理完成
696.7022964954376
第32条处理完成
719.8715827465057
第33条处理完成
744.7117838859558
第34条处理完成
772.6002042293549
第35条处理完成
784.7906579971313
第36条处理完成
804.0085623264313
第37条处理完成
823.358440160751

In [ ]:
df_sample['Conclusion'].head()

0    　　核心结论：我们预计：公司有望在2025 年-2027 年分别实现归母净利润8.18亿元、...
1    　　公司深耕电力能源领域，聚焦数字能源战略成效初显。公司成立于2002 年，是一家深耕电力能...
2    　　业内稀缺一体化供应链解决方案领军企业，下游优质客户增厚公司业绩弹性。公司长期聚焦服务于消...
3    　　公司基本情况：公司成立于2002 年，自设立以来一直专注于车用胶管系列产品，主营业务包括...
4    　　金力永磁是全球领先的高性能稀土永磁材料龙头企业。纵向来看，公司产能从2021年的1.5 ...
Name: Conclusion, dtype: object

In [27]:
final_result

['核心结论：我们预测，公司将在2025年至2027年分别实现归母净利润12.45亿元、14.82亿元、17.56亿元，对应现时市盈率为22.18x、18.63x、15.72x。首次覆盖，给予“增持”评级。\n\n公司作为国内领先的智能安防系统及城市数字化解决方案供应商，专注于智慧城市、交通、社区等公共安全领域。公司以视频监控物联网平台为核心，构建了完整的软硬件产品矩阵，并逐步拓展至智能传感器、数据分析软件及云服务，助力公共安全客户在智慧城市建设中提升运营效率和安全水平。\n\n下游市场空间广阔，公司与政府及大型企业保持稳定合作关系：根据行业数据，全球智能安防市场在智慧城市推动下持续扩张，预计年复合增长率超过10%，中国市场渗透率逐步提升至约20%。公司聚焦公共安全领域，通过与地方政府和大型企业的长期合作，积累了丰富的项目经验，在智慧安防领域形成了一定的技术壁垒和客户粘性。稳定的合作关系有助于公司维持订单连续性，支撑业务实现稳健增长。\n\n公司自主研发核心AI算法平台SmartVision，软硬件一体化优势突出：公司开发的SmartVision平台集成多种智能分析功能，具备高精度识别、实时响应和低延迟等特性。凭借在人工智能和物联网领域的技术积累，公司以软硬件协同方式，精准满足公共安全客户对智能监控和数据分析的需求，有望在智慧城市浪潮中进一步巩固市场地位，推动业务持续扩张。\n\n同时，作为智能安防行业的领军企业，公司有望在5G和物联网技术普及下，为城市安全管理提供创新解决方案，加速行业智能化升级。\n\n风险提示：政府预算支出不及预期；技术迭代风险；市场竞争加剧；宏观经济波动影响。',
 '公司专注智能制造领域，聚焦工业自动化战略成果显著。公司创立于2005年，作为工业自动化整体解决方案提供商，深耕高端装备制造近二十年，已形成工业机器人、智能物流系统两大核心业务板块。24年工业机器人业务收入占比达68.5%，成为公司主要收入支柱。公司凭借自主可控的核心技术，在汽车制造领域率先实现全流程自动化解决方案，是国内少数掌握工业机器人全产业链技术的企业之一。24年公司实现营业收入32.15亿元，归母净利润1.2亿元，同比增长128.3%，业绩实现大幅改善。\n\n工业机器人板块：全产业链协同发展，构建智能制造完整生态。①随着制造业转型升级加速，工业机器人密度持续提升。202

In [ ]:
# 批量处理代码
n_sim = 1000

# 读取数据s
final_result=[]
import time
start_time=time.time()
# 添加延迟控制，避免速率限制
for i in range(min(n_sim, len(df_sample))):
    result = call_deepseek(df_sample['Conclusion'][i],prompt_3)
    final_result.append(result)
    
    now=time.time()
    # 处理结果，这里可以根据需要保存结果
    if result:
        print(f"第{i+1}条处理完成")
    else:
        print(f"第{i+1}条处理失败")
    print(now-start_time)
    # 添加延迟避免触发速率限制
    if (i + 1) % 10 == 0:
        time.sleep(1)
    
    if i%20==0:
        pd.DataFrame(final_result).to_excel(fr'gen_1027{time.time()}.xlsx')

In [29]:
pd.DataFrame(final_result).to_excel('gen_1026.xlsx')